https://github.com/playideas/python-book/blob/main/examples/vol2/05-autograd/b04_partial.py

이 코드는 **두 변수 함수의 기울기를 중앙 유한차분으로 근사한 뒤, 그 기울기의 반대 방향으로 한 걸음 이동해 높이가 실제로 낮아지는지 확인**합니다. 여기서 `h`와 `걸음`은 서로 다른 역할을 합니다.

함수는
\[
f(x,y)=x^2+3y^2
\]
이고, `x`만 바꿔 보는 편미분은
\[
\frac{\partial f}{\partial x}=2x,
\]
`y`만 바꿔 보는 편미분은
\[
\frac{\partial f}{\partial y}=6y
\]
입니다. 현재 점 \((1,2)\)에서 기울기는 \((2,12)\)예요.

### `h`: 기울기를 재기 위한 시험 간격

코드는 각 축에서 점을 조금 앞뒤로 옮겨 함수값을 비교합니다.

\[
\frac{\partial f}{\partial x}(x,y)
\approx
\frac{f(x+h,y)-f(x-h,y)}{2h}.
\]

`y`도 똑같은 방식으로 계산하되, 그때는 `x`를 고정합니다. 이것이 **중앙 유한차분**이고, 일반적인 매끄러운 함수에서는 근사 오차가 \(O(h^2)\)입니다. <citation src="2"></citation>

이 예제에서는 더 좋은 점이 있습니다. 함수가 2차식이라서 실수 연산을 가정하면 중앙차분의 식이 정확히 정리됩니다.

\[
\frac{(x+h)^2-(x-h)^2}{2h}=2x.
\]

`y`에 대해서도 똑같이 정리되어 \(6y\)가 됩니다. 즉, 여기서 수학적인 근사 오차는 0이에요. 다만 실제 컴퓨터에서는 부동소수점 반올림이 있으므로 출력은 내부적으로 약간 흔들릴 수 있습니다.

### `걸음`: 실제로 이동할 크기

기울기 \((2,12)\)는 높이가 가장 빠르게 **증가하는** 방향입니다. 그러므로 반대 방향으로 이동하면 내려갑니다.

코드의 이동은 다음과 같습니다.

\[
(x,y)\mapsto(x,y)-0.01(2,12)=(0.98,1.88).
\]

이동 후 높이는

\[
0.98^2+3(1.88)^2=11.5636
\]

이고, 원래 높이는 \(1^2+3(2^2)=13\)입니다. 반대로 기울기 방향으로 가면 \((1.02,2.12)\)가 되어 높이가 \(14.5236\)으로 올라갑니다.

따라서 출력은 대략 다음과 같습니다.

```text
동서 방향 기울기: 2.0000  (공식 2x = 2.0)
남북 방향 기울기: 12.0000  (공식 6y = 12.0)
지금 높이: 13.0000
기울기 반대로 한 걸음: 11.5636
기울기 방향으로 한 걸음: 14.5236
```

중요한 구분은 `h=1e-5`는 **기울기를 측정할 때 함수에 넣어 보는 간격**이고, `걸음=0.01`은 **기울기를 구한 다음 실제 좌표를 업데이트하는 학습률**이라는 점입니다. 서로 바꿔 쓸 수 있는 값이 아닙니다. 또 기울기에 `0.01`을 곱했으므로 이동 거리가 정확히 `0.01`인 것도 아닙니다. 이동 벡터는 \((-0.02,-0.12)\)입니다. 경사하강법은 보통 이처럼 기울기 크기까지 반영해 이동합니다.

이 함수에 경사하강법을 반복 적용하면 업데이트는

\[
x_{k+1}=(1-2\eta)x_k,\qquad
y_{k+1}=(1-6\eta)y_k
\]

가 됩니다. 여기서 \(\eta\)는 `걸음`에 해당하는 학습률입니다. `0.01`을 쓰면 `x`는 매번 `0.98`배, `y`는 `0.94`배가 되어 둘 다 0에 가까워지고, 함수의 최솟값인 \(f(0,0)=0\)을 향합니다. 이 예제에서는 \(0<\eta<1/3\)일 때 원점으로 수렴합니다. 너무 큰 학습률은 한 번에 지나쳐 진동하거나 발산할 수 있어요.

마지막으로, 파일의 맥락이 자동 미분이어도 **이 코드는 자동 미분을 하는 코드는 아닙니다.** 주변 점에서 함수값을 직접 계산해 미분값을 추정하는 수치 미분 예제입니다. 자동 미분은 연산 과정에 미분 규칙을 적용해 기울기를 계산하며, 이 작은 예제에서는 수치 미분 결과를 정확한 공식 \(2x,6y\)와 비교해 감을 잡는 용도로 볼 수 있습니다.

In [ ]:
# [2권 Chapter 5. 자동 미분과 경사하강법 · 5.1 기울기: 조금 움직였을 때 얼마나 변하나 · manuscript-vol2/chapters/05-autograd.md:152]
def 높이(x: float, y: float) -> float:
    """두 방향(동서 x, 남북 y)으로 펼쳐진 언덕의 높이."""
    return x**2 + 3 * y**2


h = 1e-5
x, y = 1.0, 2.0
동서_기울기 = (높이(x + h, y) - 높이(x - h, y)) / (
    2 * h
)  # y는 그대로 두고 x만 움직이기
남북_기울기 = (높이(x, y + h) - 높이(x, y - h)) / (
    2 * h
)  # x는 그대로 두고 y만 움직이기
print(f"동서 방향 기울기: {동서_기울기:.4f}  (공식 2x = {2 * x})")
print(f"남북 방향 기울기: {남북_기울기:.4f}  (공식 6y = {6 * y})")

# 기울기 반대 방향으로 아주 조금 걸어 보면 높이가 내려가는지 확인
걸음 = 0.01
print(f"지금 높이: {높이(x, y):.4f}")
print(
    f"기울기 반대로 한 걸음: {높이(x - 걸음 * 동서_기울기, y - 걸음 * 남북_기울기):.4f}"
)
print(
    f"기울기 방향으로 한 걸음: {높이(x + 걸음 * 동서_기울기, y + 걸음 * 남북_기울기):.4f}"
)


---

핵심부터 찢으면, 이 링크들은 전부 “비슷한가?”를 다루는 것처럼 보여도 **같은 질문에 답하는 지표가 아닙니다.** 방향을 비교하는 지표, 좌표상 거리를 재는 지표, 예측 오차에 벌점을 매기는 지표를 한데 섞으면 점수 해석이 꼬입니다.

## 네 지표가 실제로 묻는 것

| 지표 | 계산 | 실제 질문 | 낮을수록/높을수록 |
|---|---|---|---|
| 코사인 유사도 | \(\frac{a\cdot b}{\|a\|\|b\|}\) | 두 벡터가 얼마나 같은 방향을 향하나? | 높을수록 유사 |
| 유클리드 거리 | \(\sqrt{\sum_i(a_i-b_i)^2}\) | 두 점이 직선으로 얼마나 떨어져 있나? | 낮을수록 가까움 |
| 맨해튼 거리 | \(\sum_i|a_i-b_i|\) | 축 방향으로만 이동하면 얼마나 가야 하나? | 낮을수록 가까움 |
| MSE | \(\frac1d\sum_i(a_i-b_i)^2\) | 좌표별 예측 오차가 평균적으로 얼마나 큰가? | 낮을수록 좋음 |

코사인 유사도는 **방향**, 유클리드·맨해튼 거리는 **위치 차이**, MSE는 **정답과 예측 사이의 오차**를 묻습니다. “유사도”는 높은 점수가 좋은 식으로 정의하는 경우가 많고, “거리”는 낮을수록 가까운 식입니다. 수학적으로 진짜 거리(metric)가 되려면 삼각부등식 같은 조건도 만족해야 합니다. <citation src="17,25"></citation>

## 같은 데이터인데 순위가 뒤집히는 예

질의 벡터가 \(q=(1,0)\), 후보가 \(a=(100,0)\), \(b=(1,1)\)이라고 해보죠.

- 코사인 유사도: \(q\)와 \(a\)는 **1.0**, \(q\)와 \(b\)는 약 **0.707**
- 유클리드 거리: \(a\)까지는 **99**, \(b\)까지는 **1**

코사인은 \(a\)를 더 비슷하다고 보고, 유클리드 거리는 \(b\)를 더 가깝다고 봅니다. 모순이 아니라 **측정 기준이 다르기 때문**입니다. \(a\)는 질의와 방향은 완전히 같지만 크기가 압도적으로 다릅니다. 코사인은 크기를 무시하고, 유클리드는 그 차이를 반영합니다.

```python
import numpy as np
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances,
    manhattan_distances,
)

q = np.array([[1.0, 0.0]])
candidates = np.array([
    [100.0, 0.0],
    [1.0, 1.0],
])

print(cosine_similarity(q, candidates))
# [[1.         0.7071...]]

print(euclidean_distances(q, candidates))
# [[99.  1.]]

print(manhattan_distances(q, candidates))
# [[99.  1.]]

print(np.mean((candidates - q) ** 2, axis=1))
# [4900.5    0.5]
```

scikit-learn의 `cosine_similarity(X, Y)`는 `X`와 `Y`의 **행을 각각 샘플**, 열을 특성으로 보고 모든 행 쌍의 점수를 반환합니다. 출력 shape은 `(X의 행 수, Y의 행 수)`예요. 수식상 \(X\)와 \(Y\)는 정규화된 내적을 계산하며, 이미 L2 정규화한 벡터라면 코사인 유사도는 내적과 같습니다. <citation src="23,25"></citation>

## 코사인 유사도에서 특히 조심할 점

**1. “코사인 거리”는 이름과 달리 항상 진짜 거리가 아닙니다.**  
흔히 코사인 거리를 \(1-\text{cosine similarity}\)로 정의하지만, 이 값은 삼각부등식을 항상 만족하지 않습니다. 게다가 방향이 같은 벡터 \(a=(1,0)\), \(b=(100,0)\)는 서로 다른 벡터인데도 코사인 거리가 0입니다. 거리의 조건인 “거리가 0이면 두 대상이 동일해야 한다”도 깨지는 셈입니다. 각도 자체인 \(\arccos(\text{cosine similarity})\)는 적절한 거리(metric)가 됩니다. <citation src="6,16"></citation>

**2. 0 벡터에는 방향이 없어서 코사인 유사도가 수학적으로 정의되지 않습니다.**  
scikit-learn 구현은 0 벡터가 들어오면 0 유사도를 돌려주는 예를 보여주지만, 이를 “두 대상이 아무 관련 없다”는 의미로 단정하면 곤란합니다. 입력 생성 과정에서 0 벡터가 생긴 건지, 실제로 비교할 수 없는 사례인지 별도로 처리하는 편이 안전합니다. <citation src="23"></citation>

**3. 크기 무시는 장점이면서 동시에 버리는 정보입니다.**  
문서 길이나 벡터 크기가 중요하지 않고 주제 방향만 중요하면 코사인이 잘 맞을 수 있습니다. 반대로 횟수, 강도, 규모 자체가 의미를 가진다면 코사인이 중요한 차이를 지워 버립니다. 내적은 방향뿐 아니라 크기도 반영하므로, 코사인과 같은 척도가 아닙니다. 벡터가 모두 단위 길이로 정규화된 경우에만 코사인 유사도와 내적이 일치합니다. <citation src="22,23"></citation>

## 유클리드·맨해튼·MSE는 서로 얼마나 다른가

유클리드 거리는 직선 거리인 L2, 맨해튼 거리는 좌표별 절댓값 차를 더하는 L1입니다. 2차원에서 유클리드 거리가 원형으로 같은 거리를 나타낸다면, 맨해튼 거리는 마름모형으로 나타납니다. 맨해튼 거리를 “택시가 격자 도로를 따라가는 거리”로 비유하는 이유죠. 어느 쪽이 더 좋은지는 데이터 좌표축과 문제에서 어떤 오차를 중요하게 보는지에 달렸습니다. 특히 단위나 특성 스케일이 다른데 그대로 거리를 재면 큰 숫자 범위의 특성이 결과를 지배할 수 있습니다. <citation src="1"></citation>

MSE는 한 쌍의 같은 차원 벡터에 대해서는

\[
\mathrm{MSE}(a,b)=\frac{\|a-b\|_2^2}{d}
\]

입니다. 여기서 \(d\)는 특성 수예요. 그래서 **같은 차원의 벡터 쌍끼리 비교할 때는** MSE가 유클리드 거리의 제곱을 \(d\)로 나눈 값입니다. 순위는 같지만 점수의 의미와 단위는 다르고, MSE는 제곱 오차라 큰 오차를 특히 강하게 벌줍니다. 예를 들어 오차가 한 좌표에서 10인 경우 그 좌표의 제곱 오차는 100입니다.

하지만 MSE는 보통 임베딩 검색의 “의미 유사도”를 재려고 쓰기보다, 회귀 모델의 예측값이 정답에서 얼마나 벗어났는지 평가하는 **손실 함수·평가 지표**로 씁니다. scikit-learn의 `mean_squared_error(y_true, y_pred)`도 정답 배열과 예측 배열의 오차를 평균내며, 여러 출력이 있으면 기본적으로 출력별 오차도 평균냅니다. MSE는 이름에 error가 들어가고 거리처럼 쓰이기도 하지만, 제곱 때문에 수학적 거리의 삼각부등식을 만족하지는 않습니다. <citation src="12,15"></citation>

## 정규화된 벡터라면 유클리드와 코사인이 연결된다

두 벡터가 모두 길이 1이면,

\[
\|u-v\|_2^2
= \|u\|^2+\|v\|^2-2u\cdot v
=2-2\cos(u,v).
\]

따라서 단위 길이 벡터끼리는 **코사인 유사도가 높은 순서와 유클리드 거리가 낮은 순서가 정확히 같습니다.** 다만 숫자 점수 자체는 같지 않습니다. 정규화되지 않은 벡터에서는 이 순위 동등성이 보장되지 않습니다. <citation src="6,7"></citation>

실전 선택은 이렇게 요약할 수 있습니다.

- **텍스트·임베딩 검색:** 방향이 핵심이면 코사인을 후보로 삼고, 모델이 학습된 방식과 실제 검색 성능으로 확인.
- **좌표·수치 특성의 절대 차이:** 크기 차이가 의미 있으면 유클리드.
- **축별 차이를 더하고 큰 오차의 영향은 제곱보다 덜 키우고 싶음:** 맨해튼을 검토.
- **회귀 예측값을 평가하거나 큰 오차를 강하게 벌주고 싶음:** MSE.

“임베딩이면 무조건 코사인”도, “거리니까 유클리드가 더 올바름”도 아닙니다. **벡터가 무엇을 인코딩하는지, 크기가 의미 있는지, 해당 모델이 어떤 척도에 맞춰 학습됐는지**가 선택 기준입니다. <citation src="1,22"></citation>